# Lesson 1A: K-Means Clustering - Theory

## Introduction

You are leading a marketing team at an e-commerce company. You have data on thousands of customers including their purchase history, browsing patterns, demographics, and engagement metrics. Your goal is to segment customers into groups for targeted marketing campaigns.

But here is the problem: You do not know how many customer segments exist or what defines them. You just know that grouping similar customers together will help you create more effective marketing strategies.

This is exactly what **K-Means clustering** solves. It is the most popular and fundamental clustering algorithm used in machine learning.

**Real-world applications**:
- **Customer segmentation**: Group customers by behavior for personalized marketing
- **Image compression**: Reduce colors in an image by clustering similar colors
- **Document organization**: Group similar documents or articles by topic
- **Anomaly detection**: Identify outliers that do not fit any cluster
- **Feature engineering**: Create cluster-based features for supervised learning

In this lesson, we will:
1. Understand the K-Means algorithm and its mathematical formulation
2. Derive Lloyd's algorithm from first principles
3. Learn about K-Means++ initialization for better results
4. Implement K-Means from scratch using NumPy
5. Analyze convergence guarantees and time complexity
6. Visualize how K-Means iteratively finds clusters

Then in Lesson 1B, we will use scikit-learn for production implementations and apply K-Means to real market segmentation problems.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from typing import Tuple, List
from numpy.typing import NDArray

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries loaded successfully!")

## The K-Means Problem

### Problem Statement

Given:
- Dataset $X = \{x^{(1)}, x^{(2)}, ..., x^{(m)}\}$ where $x^{(i)} \in \mathbb{R}^n$
- Number of clusters $k$

**Goal**: Find $k$ cluster centers $\mu = \{\mu_1, \mu_2, ..., \mu_k\}$ that minimize the **within-cluster sum of squares** (WCSS):

$$J(C, \mu) = \sum_{i=1}^{k} \sum_{x \in C_i} \|x - \mu_i\|^2$$

where:
- $C_i$ is the set of points assigned to cluster $i$
- $\mu_i$ is the centroid (mean) of cluster $i$
- $\|x - \mu_i\|^2$ is the squared Euclidean distance

### Intuition

We want to find clusters where:
1. **Points within a cluster are close to their centroid** (low intra-cluster distance)
2. **Points in different clusters are far apart** (high inter-cluster distance)

Minimizing WCSS achieves both goals simultaneously.

## Lloyd's Algorithm

The standard K-Means algorithm is called **Lloyd's algorithm** (1957). It uses an iterative approach with two alternating steps.

### Algorithm Steps

**Input**: Dataset $X$, number of clusters $k$

**Output**: Cluster assignments $C$ and centroids $\mu$

1. **Initialize**: Randomly select $k$ points from $X$ as initial centroids $\mu_1, ..., \mu_k$

2. **Repeat until convergence**:

   a. **Assignment Step**: Assign each point to nearest centroid
   $$C_i = \{x^{(j)} : \|x^{(j)} - \mu_i\| \leq \|x^{(j)} - \mu_l\| \text{ for all } l = 1,...,k\}$$

   b. **Update Step**: Recompute centroids as mean of assigned points
   $$\mu_i = \frac{1}{|C_i|} \sum_{x \in C_i} x$$

3. **Convergence**: Stop when centroids no longer change (or change is below threshold)

### Why Does This Work?

**Key insight**: Each step decreases (or keeps constant) the cost function $J(C, \mu)$

- **Assignment step**: For fixed centroids, we minimize cost by assigning each point to its nearest centroid
- **Update step**: For fixed assignments, we minimize cost by setting centroids to cluster means

This is a **coordinate descent** algorithm that alternates between optimizing assignments and centroids.

**Proof that update step minimizes cost**:

For cluster $C_i$, we want to minimize:
$$\sum_{x \in C_i} \|x - \mu_i\|^2$$

Taking the gradient with respect to $\mu_i$ and setting to zero:
$$\frac{\partial}{\partial \mu_i} \sum_{x \in C_i} \|x - \mu_i\|^2 = -2 \sum_{x \in C_i} (x - \mu_i) = 0$$

Solving:
$$\sum_{x \in C_i} x = |C_i| \mu_i$$
$$\mu_i = \frac{1}{|C_i|} \sum_{x \in C_i} x$$

This proves the centroid should be the mean of its cluster.

In [ ]:
class KMeans:
    """
    K-Means clustering implementation from scratch.

    K-Means partitions n observations into k clusters where each observation
    belongs to the cluster with the nearest mean (centroid).
    """

    def __init__(self, n_clusters: int = 3, max_iters: int = 100,
                 tol: float = 1e-4, random_state: int = 42):
        """
        Initialize K-Means clustering.

        Args:
            n_clusters: Number of clusters to form
            max_iters: Maximum number of iterations
            tol: Tolerance for convergence (centroid movement threshold)
            random_state: Random seed for reproducibility
        """
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.tol = tol
        self.random_state = random_state
        self.centroids = None
        self.labels = None
        self.inertia_ = None  # WCSS (within-cluster sum of squares)
        self.n_iter_ = 0  # Number of iterations run

    def fit(self, X: NDArray) -> 'KMeans':
        """
        Compute K-Means clustering.

        Args:
            X: Training data of shape (n_samples, n_features)

        Returns:
            self: Fitted estimator
        """
        np.random.seed(self.random_state)

        n_samples, n_features = X.shape

        # Initialize centroids by randomly selecting k points from dataset
        random_indices = np.random.choice(n_samples, self.n_clusters, replace=False)
        self.centroids = X[random_indices].copy()

        for iteration in range(self.max_iters):
            # Assignment step: Assign each point to nearest centroid
            labels = self._assign_clusters(X)

            # Update step: Recompute centroids
            new_centroids = self._compute_centroids(X, labels)

            # Check for convergence
            centroid_shift = np.linalg.norm(new_centroids - self.centroids)
            self.centroids = new_centroids
            self.n_iter_ = iteration + 1

            if centroid_shift < self.tol:
                break

        # Final assignment
        self.labels = self._assign_clusters(X)

        # Compute final inertia (WCSS)
        self.inertia_ = self._compute_inertia(X, self.labels, self.centroids)

        return self

    def _assign_clusters(self, X: NDArray) -> NDArray:
        """
        Assign each point to the nearest centroid.

        Args:
            X: Data of shape (n_samples, n_features)

        Returns:
            labels: Cluster assignment for each sample of shape (n_samples,)
        """
        # Compute distances from each point to each centroid
        # Shape: (n_samples, n_clusters)
        distances = np.sqrt(((X[:, np.newaxis] - self.centroids) ** 2).sum(axis=2))

        # Assign each point to nearest centroid
        labels = np.argmin(distances, axis=1)

        return labels

    def _compute_centroids(self, X: NDArray, labels: NDArray) -> NDArray:
        """
        Compute cluster centroids as mean of assigned points.

        Args:
            X: Data of shape (n_samples, n_features)
            labels: Cluster assignments of shape (n_samples,)

        Returns:
            centroids: New centroid positions of shape (n_clusters, n_features)
        """
        n_features = X.shape[1]
        centroids = np.zeros((self.n_clusters, n_features))

        for k in range(self.n_clusters):
            # Get all points assigned to cluster k
            cluster_points = X[labels == k]

            if len(cluster_points) > 0:
                # Compute mean of points in this cluster
                centroids[k] = cluster_points.mean(axis=0)
            else:
                # If cluster is empty, reinitialize randomly
                centroids[k] = X[np.random.choice(len(X))]

        return centroids

    def _compute_inertia(self, X: NDArray, labels: NDArray, centroids: NDArray) -> float:
        """
        Compute within-cluster sum of squares (WCSS).

        Args:
            X: Data of shape (n_samples, n_features)
            labels: Cluster assignments of shape (n_samples,)
            centroids: Centroid positions of shape (n_clusters, n_features)

        Returns:
            inertia: Total WCSS across all clusters
        """
        inertia = 0.0
        for k in range(self.n_clusters):
            cluster_points = X[labels == k]
            if len(cluster_points) > 0:
                inertia += np.sum((cluster_points - centroids[k]) ** 2)

        return inertia

    def predict(self, X: NDArray) -> NDArray:
        """
        Predict cluster labels for new data.

        Args:
            X: New data of shape (n_samples, n_features)

        Returns:
            labels: Predicted cluster for each sample
        """
        return self._assign_clusters(X)

print("✅ K-Means class implemented successfully!")

In [ ]:
# Test K-Means on synthetic data
print("Creating synthetic dataset with 3 clear clusters...")

# Generate 3 clusters of points
X, y_true = make_blobs(n_samples=300, centers=3, n_features=2,
                       cluster_std=0.5, random_state=42)

# Fit K-Means
print("\nFitting K-Means with k=3...")
kmeans = KMeans(n_clusters=3, max_iters=100, random_state=42)
kmeans.fit(X)

print(f"\n✅ K-Means converged in {kmeans.n_iter_} iterations")
print(f"Final inertia (WCSS): {kmeans.inertia_:.2f}")
print(f"\nCentroid positions:")
for i, centroid in enumerate(kmeans.centroids):
    print(f"  Cluster {i}: {centroid}")

# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot true clusters
axes[0].scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', alpha=0.6, edgecolors='k')
axes[0].set_title('True Clusters', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].grid(True, alpha=0.3)

# Plot K-Means clusters
axes[1].scatter(X[:, 0], X[:, 1], c=kmeans.labels, cmap='viridis', alpha=0.6, edgecolors='k')
axes[1].scatter(kmeans.centroids[:, 0], kmeans.centroids[:, 1],
                c='red', marker='X', s=300, edgecolors='black', linewidths=2,
                label='Centroids')
axes[1].set_title(f'K-Means Clusters (k={kmeans.n_clusters})', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ K-Means successfully recovered the true clusters!")